# DeepGlobe Land-Cover Segmentation — Live Demo

Two interchangeable deliverables:

- **Gradio demo (section 4)** — interactive browser UI, great for live Q&A.
- **Pre-rendered MP4 (section 5)** — deterministic backup you can embed in slides.

Both are backed by the project's best model
(**DeepLabV3+-R50 + Combined CE+Dice**, test mIoU 0.7332).

**Prerequisites** — a trained checkpoint on your Google Drive. The default
path below assumes `MyDrive/landcover/checkpoints/deeplab_r50_combined_best.pt`;
edit it to match wherever your weights live.

**Runtime** — select `Runtime -> Change runtime type -> GPU` (T4 is fine).
Inference runs in ~1 s per 2448&sup2; image on a T4, ~0.8 s on an A100.

## 1 &middot; Clone the repo and install demo dependencies

In [ ]:
%cd /content
![ -d landcover-seg ] || git clone https://github.com/Jerryzhang258/landcover-seg
%cd /content/landcover-seg
!pip install -q segmentation-models-pytorch==0.3.3 albumentations==1.3.1 gradio==4.29.0

## 2 &middot; Mount Google Drive and pick the checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# EDIT ME: path to your trained checkpoint on Drive.
CKPT_PATH = '/content/drive/MyDrive/landcover/checkpoints/deeplab_r50_combined_best.pt'
MODEL_NAME = 'deeplab_r50'  # one of: deeplab_r50, deeplab_r101, unet_resnet34, attn_unet, unet_scratch, vanilla_unet

import os
assert os.path.isfile(CKPT_PATH), f'checkpoint not found: {CKPT_PATH}'
print(f'OK — will load {MODEL_NAME} from {CKPT_PATH}')

## 3 &middot; (Optional) stage a few example images

Anything you drop into `demo/examples/` becomes a one-click button in the UI,
and any of those files can be passed to the MP4 recorder in section 5.
If you kept the DeepGlobe test split on Drive, copy 3-5 id's across.

In [ ]:
# EDIT ME (optional): source directory of raw DeepGlobe images on Drive.
SRC_IMG_DIR  = '/content/drive/MyDrive/landcover/data/raw/train'
EXAMPLE_IDS  = []  # e.g. ['104876', '153214', '629571']

import shutil, glob
os.makedirs('demo/examples', exist_ok=True)
for iid in EXAMPLE_IDS:
    for path in glob.glob(f'{SRC_IMG_DIR}/{iid}_sat.jpg'):
        shutil.copy(path, f'demo/examples/{iid}.jpg')
print('examples:', sorted(os.listdir('demo/examples')))

## 4 &middot; Launch the Gradio demo

The cell below launches the UI and prints two URLs:

- `http://127.0.0.1:7860` - only reachable from this notebook.
- `https://*.gradio.live` - **this is the one to open in your presentation browser**.
  It's valid for 72 hours and can be opened from any device.

Interrupt this cell (`Runtime -> Interrupt execution`) to shut the demo down.

Skip this section entirely if you only need the MP4 from section 5.

In [ ]:
!python demo/app.py --ckpt $CKPT_PATH --model $MODEL_NAME --share

## 5 &middot; Record a backup MP4

Produces a deterministic 1920&times;1080 video that stitches together
*Input &rarr; Prediction &rarr; Overlay* panels plus a per-class pixel-proportion
bar chart for every image you list below. Drop the output into your slides
as a backup in case the live Gradio tunnel misbehaves.

Pick 3-5 images that **span your three findings**: one mixed-urban scene,
one barren-heavy scene, one rangeland-heavy scene (showcases the known
Rangeland&rarr;Agriculture confusion from the report).

In [ ]:
# EDIT ME: the images you want in the video, in order of appearance.
VIDEO_IMAGES = [
    # f'{SRC_IMG_DIR}/104876_sat.jpg',
    # f'{SRC_IMG_DIR}/153214_sat.jpg',
    # f'{SRC_IMG_DIR}/629571_sat.jpg',
]
VIDEO_OUT     = 'demo/demo_video.mp4'
POSTER_OUT    = 'demo/demo_poster.png'
SECONDS_EACH  = 5.0

assert VIDEO_IMAGES, 'add at least one image path to VIDEO_IMAGES and re-run'
missing = [p for p in VIDEO_IMAGES if not os.path.isfile(p)]
assert not missing, f'these images were not found: {missing}'

img_args = ' '.join(f'"{p}"' for p in VIDEO_IMAGES)
!python demo/record_video.py \
    --ckpt "$CKPT_PATH" \
    --model "$MODEL_NAME" \
    --images $img_args \
    --out "$VIDEO_OUT" \
    --poster "$POSTER_OUT" \
    --seconds-per-image $SECONDS_EACH

In [ ]:
# Preview the rendered video inline and offer a download button.
from IPython.display import HTML
import base64
with open(VIDEO_OUT, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
HTML(f'''
<video width="960" controls>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
<p>File: <code>{VIDEO_OUT}</code> &middot; right-click &rarr; Save video as... to download.</p>
''')

In [ ]:
# (Optional) copy the video back to Drive so you can pick it up on your laptop.
import shutil
DRIVE_OUT_DIR = '/content/drive/MyDrive/landcover/demo'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
shutil.copy(VIDEO_OUT, DRIVE_OUT_DIR)
shutil.copy(POSTER_OUT, DRIVE_OUT_DIR)
print(f'copied to {DRIVE_OUT_DIR}')

---

### Tips for the talk

1. **Pre-upload 3-5 curated images** via `demo/examples/` so you never
   depend on a drag-and-drop during the live click-through.
2. **Always record the MP4 backup** (section 5) even if you plan to run
   the Gradio demo live. Slide embedding is free insurance against
   Colab timeouts, Wi-Fi flakiness, or share-tunnel hiccups.
3. **Pick images that showcase the three findings**: one mixed urban-rural
   scene (overall accuracy), one barren-heavy scene (loss-rebalancing gain),
   one rangeland-heavy scene (the known failure mode).
4. **Keep the browser tab already loaded** - Gradio's first cold start
   takes ~10 s while the share tunnel comes up.